# Multivariate Regression, DATA MANIPULATION, and ANALYSIS

## Gather all gage characteristics I will be using for the multivariable regression

### Import all necessary functions

In [1]:
# Load a
import sys
import os as os

import geopandas as gpd
import numpy as np
from numpy.polynomial import Polynomial
import psycopg2
from netCDF4 import Dataset

from tqdm import tqdm
from multiprocessing import Pool
import statsmodels.api as sm
from scipy import stats
# from scipy import optimize

import cartopy.crs as ccrs
from cartopy.feature import NaturalEarthFeature as cfNEF

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import patches
#import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D

import rasterio
import xarray as xr

import random
import subprocess
import netCDF4 as nc
import shutil

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import ShuffleSplit
import seaborn as sns
from scipy.stats import gaussian_kde

import pathlib
import glob
import pandas as pd

from sklearn.metrics import r2_score
from scipy.stats import normaltest

import geopandas as gpd
import xarray as xr
from rasterio import features
from shapely.geometry import mapping
import rioxarray
import scipy


import geopandas as gpd
import rioxarray
from shapely.geometry import mapping
from scipy import stats

### Load all shapefiles from the 3 assimilations and concatinate into one

In [2]:
# Load shapefile that contains stations and delta SWE from the January 10th assimilation
shapefile_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Test_Runs\01_10_24_Assimilations\snodas_assim_20240110_with_umrb\ssm1054_md_based_2024010912_2024011012_central.shp"

# Load shapefile
Jan_10_2024_stations = gpd.read_file(shapefile_path)

# Drop all points that arent stations
Jan_10_2024_stations = Jan_10_2024_stations[Jan_10_2024_stations['VALUE'] != 1]

# Just keep the columns Latitude, Longitude, Station_id, station_ty, station_elevation, DEM_elevation, Forest_den, D_SWE_OM
# Keep only the specified columns
columns_to_keep = ['LATITUDE', 'LONGITUDE', 'STATION_ID', 'STATION_TY', 'STATION_EL', 'DEM_ELEVAT', 'FOREST_DEN', 'MD_DEPTH_T','D_SWE_OM', 'OB_SWE', 'MD_SWE' ]

Jan_10_2024_stations = Jan_10_2024_stations[columns_to_keep]

print(Jan_10_2024_stations)

      LATITUDE  LONGITUDE        STATION_ID STATION_TY  STATION_EL  \
3401   43.1653   -95.1467               3SE    COOPABC       403.0   
3402   41.4461   -83.3289  41.4461_083.3289    SPOTTER       190.0   
3403   41.6383   -83.5281  41.6383_083.5281    SPOTTER       187.0   
3404   45.6708   -96.9961               8D3       ASOS       353.0   
3405   42.2416   -83.6933             AASM4     COOPAB       254.0   
...        ...        ...               ...        ...         ...   
7053   47.2719  -100.3303             WNGN8       UMRB     -9999.0   
7054   43.3100   -99.6900             WNMS2       UMRB     -9999.0   
7055   43.5600  -100.7400             WRIS2       UMRB     -9999.0   
7056   43.7322   -98.6939             WTES2       UMRB     -9999.0   
7057   46.0135   -99.6876             ZELN8       UMRB     -9999.0   

      DEM_ELEVAT  FOREST_DEN      MD_DEPTH_T  D_SWE_OM    OB_SWE    MD_SWE  
3401       405.0         3.0   2024-01-10 06  0.009460  0.014676  0.005216  
3402 

In [3]:
# Load shapefile that contains stations and delta SWE from the 12/11/22 assimilation
shapefile_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Test_Runs\12_11_22_Assimilations\snodas_assim_20221211_with_umrb\ssm1054_md_based_2022121012_2022121112_east.shp"

# Load shapefile
Dec_11_2022_stations = gpd.read_file(shapefile_path)

# Drop all points that arent stations
Dec_11_2022_stations = Dec_11_2022_stations[Dec_11_2022_stations['VALUE'] != 1]

# Just keep the columns Latitude, Longitude, Station_id, station_ty, station_elevation, DEM_elevation, Forest_den, D_SWE_OM
# Keep only the specified columns
columns_to_keep = ['LATITUDE', 'LONGITUDE', 'STATION_ID', 'STATION_TY', 'STATION_EL', 'DEM_ELEVAT', 'FOREST_DEN','MD_DEPTH_T', 'D_SWE_OM', 'OB_SWE', 'MD_SWE']

Dec_11_2022_stations = Dec_11_2022_stations[columns_to_keep]

print(Dec_11_2022_stations)

       LATITUDE   LONGITUDE  STATION_ID  \
3042  46.993439  -65.506909   CAN-NB-11   
3043  47.289956  -68.411945  CAN-NB-134   
3044  47.057010  -67.726300   CAN-NB-38   
3045  46.868300  -68.013600         CAR   
3046  46.868100  -68.012500       CARM1   
...         ...         ...         ...   
4136  44.088600 -103.299700     SD-PN-2   
4137  44.086570 -103.370100    SD-PN-40   
4138  44.497300 -103.871700       SPES2   
4139  44.424500 -103.379300       STMS2   
4140  44.072500 -103.212200       UNRS2   

                                       STATION_TY  STATION_EL  DEM_ELEVAT  \
3042                                     COCORAHS        33.0        28.0   
3043                                     COCORAHS       147.0       152.0   
3044                                     COCORAHS       200.0       180.0   
3045  ARSR4, ASOS, AWIPS, COOPAB, CRS, UA, WSR88D       187.0       184.0   
3046                               COOPAB, SNOCOR       191.0       184.0   
...                  

In [4]:
# Load shapefile that contains stations and delta SWE from the 12/28/23 assimilation
shapefile_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Test_Runs\12_28_23_Assimilations\snodas_assim_20231228_with_umrb\ssm1054_md_based_2023122712_2023122812_central.shp"

# Load shapefile
Dec_28_2023_stations = gpd.read_file(shapefile_path)

# Drop all points that arent stations
Dec_28_2023_stations  = Dec_28_2023_stations [Dec_28_2023_stations ['VALUE'] != 1]

# Just keep the columns Latitude, Longitude, Station_id, station_ty, station_elevation, DEM_elevation, Forest_den, D_SWE_OM
# Keep only the specified columns
columns_to_keep = ['LATITUDE', 'LONGITUDE', 'STATION_ID', 'STATION_TY', 'STATION_EL', 'DEM_ELEVAT', 'FOREST_DEN', 'MD_DEPTH_T','D_SWE_OM', 'OB_SWE', 'MD_SWE']

Dec_28_2023_stations  = Dec_28_2023_stations [columns_to_keep]

print(Dec_28_2023_stations )

      LATITUDE  LONGITUDE STATION_ID                             STATION_TY  \
1707   43.1653   -95.1467        3SE                                COOPABC   
1708   45.6708   -96.9961        8D3                                   ASOS   
1709   45.4553   -98.4141      ABES2                                COOPABC   
1710   45.4825   -87.8119      ABGW3                                  UCOOP   
1711   45.4558   -98.4128        ABR  ASOS, AWIPS, COOPABC, CRS, UA, WSR88D   
...        ...        ...        ...                                    ...   
3345   47.1472  -108.5913      WRHM8                                   UMRB   
3346   43.5600  -100.7400      WRIS2                                   UMRB   
3347   44.0026  -107.9050      WRMW4                                   UMRB   
3348   43.7322   -98.6939      WTES2                                   UMRB   
3349   46.0135   -99.6876      ZELN8                                   UMRB   

      STATION_EL  DEM_ELEVAT  FOREST_DEN      MD_DE

In [5]:
# Look at UMRB Station attribute text file from Carrie
umrb_station_attributes = pd.read_csv(r'C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\umrb_meta_20240425.txt',delimiter='|',header=None, engine='python')

# Keep only the first 8 columns
umrb_station_attributes = umrb_station_attributes.iloc[:, :8]

# Rename the columns
umrb_station_attributes.columns = ['STATION_ID', 'LOCATION', 'UNKNOWN', 'STATION_TY', 'LATITUDE', 'LONGITUDE', 'STATION_EL', 'DEM_ELEVAT']

print(umrb_station_attributes)

       STATION_ID                           LOCATION   UNKNOWN  \
0     FRGM8         FORESTGROVE N                      TFX       
1     PLMW4         POWELL 2SW - WY MESONET            RIW       
2     AIDN8         AMIDON NDAWN                       BIS       
3     RAYN8         RAY 4N NDAWN                       BIS       
4     BOMN8         BOWMAN NDAWN                       BIS       
..            ...                                ...       ...   
107   WIEM8         WINNETT 7SW                        GGW       
108   WRHM8         WAR HORSE NW                       GGW       
109   DOOM8         DOOLEY NDAWN                       GGW       
110   RSTM8         REDSTONE NDAWN                     GGW       
111   FROM8         FROID NDAWN                        GGW       

          STATION_TY  LATITUDE  LONGITUDE  STATION_EL  DEM_ELEVAT  
0     MESO-UMRB        47.1057  -109.0888        1243        1251  
1     MESO-ST          44.7764  -108.7590        1333        1333  
2  

### Add windspeed into attribute dataframe

In [6]:
# Upload 2024 wind speed data

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\vs_2024.nc"

#read in the file with xarray:
wind_speed_2024 = xr.open_dataset(file_path)

wind_speed_2024 = wind_speed_2024.compute()

wind_speed_2024

<xarray.Dataset> Size: 1GB
Dimensions:     (lon: 1386, lat: 585, day: 190, crs: 1)
Coordinates:
  * lon         (lon) float64 11kB -124.8 -124.7 -124.7 ... -67.14 -67.1 -67.06
  * lat         (lat) float64 5kB 49.4 49.36 49.32 49.28 ... 25.15 25.11 25.07
  * day         (day) datetime64[ns] 2kB 2024-01-01 2024-01-02 ... 2024-07-08
  * crs         (crs) uint16 2B 3
Data variables:
    wind_speed  (day, lat, lon) float64 1GB nan nan nan nan ... nan nan nan nan
Attributes: (12/22)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    last_permanent_slice:       130
    last_early_slice:           190
    last_provisional_slice:     184
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [7]:
# Upload 2022 wind speed data

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\vs_2022.nc"

#read in the file with xarray:
wind_speed_2022 = xr.open_dataset(file_path)

wind_speed_2022 = wind_speed_2022.compute()

wind_speed_2022

<xarray.Dataset> Size: 2GB
Dimensions:     (lon: 1386, lat: 585, day: 365, crs: 1)
Coordinates:
  * lon         (lon) float64 11kB -124.8 -124.7 -124.7 ... -67.14 -67.1 -67.06
  * lat         (lat) float64 5kB 49.4 49.36 49.32 49.28 ... 25.15 25.11 25.07
  * day         (day) datetime64[ns] 3kB 2022-01-01 2022-01-02 ... 2022-12-31
  * crs         (crs) uint16 2B 3
Data variables:
    wind_speed  (day, lat, lon) float64 2GB nan nan nan nan ... nan nan nan nan
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       15 January 2023
    note1:                      The projection information for this file is: ...
    note2:                      Citation: Abatzoglou, J.T., 2013, Development...
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [8]:
# Upload 2023 wind speed data

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\vs_2023.nc"

#read in the file with xarray:
wind_speed_2023 = xr.open_dataset(file_path)

wind_speed_2023 = wind_speed_2023.compute()

wind_speed_2023

<xarray.Dataset> Size: 2GB
Dimensions:     (lon: 1386, lat: 585, day: 365, crs: 1)
Coordinates:
  * lon         (lon) float64 11kB -124.8 -124.7 -124.7 ... -67.14 -67.1 -67.06
  * lat         (lat) float64 5kB 49.4 49.36 49.32 49.28 ... 25.15 25.11 25.07
  * day         (day) datetime64[ns] 3kB 2023-01-01 2023-01-02 ... 2023-12-31
  * crs         (crs) uint16 2B 3
Data variables:
    wind_speed  (day, lat, lon) float64 2GB nan nan nan nan ... nan nan nan nan
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       02 March 2024
    note1:                      The projection information for this file is: ...
    note2:                      Citation: Abatzoglou, J.T., 2013, Development...
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [9]:
# Initialize an empty list to store wind speeds
wind_speeds = []

# Extracting data based on lat, lon, and time from Jan_10_2024_stations
for index, (lat, lon, d_str) in Jan_10_2024_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    wind_speed = wind_speed_2024.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['wind_speed'].item()
    
    # Append wind speed to the list
    wind_speeds.append(wind_speed)

# Add 'wind_speed' column to Jan_10_2024_stations
Jan_10_2024_stations['wind_speed'] = wind_speeds

# Display the updated DataFrame
print(Jan_10_2024_stations)

      LATITUDE  LONGITUDE        STATION_ID STATION_TY  STATION_EL  \
3401   43.1653   -95.1467               3SE    COOPABC       403.0   
3402   41.4461   -83.3289  41.4461_083.3289    SPOTTER       190.0   
3403   41.6383   -83.5281  41.6383_083.5281    SPOTTER       187.0   
3404   45.6708   -96.9961               8D3       ASOS       353.0   
3405   42.2416   -83.6933             AASM4     COOPAB       254.0   
...        ...        ...               ...        ...         ...   
7053   47.2719  -100.3303             WNGN8       UMRB     -9999.0   
7054   43.3100   -99.6900             WNMS2       UMRB     -9999.0   
7055   43.5600  -100.7400             WRIS2       UMRB     -9999.0   
7056   43.7322   -98.6939             WTES2       UMRB     -9999.0   
7057   46.0135   -99.6876             ZELN8       UMRB     -9999.0   

      DEM_ELEVAT  FOREST_DEN      MD_DEPTH_T  D_SWE_OM    OB_SWE    MD_SWE  \
3401       405.0         3.0   2024-01-10 06  0.009460  0.014676  0.005216   
340

In [10]:
# Initialize an empty list to store wind speeds
wind_speeds = []

# Extracting data based on lat, lon, and time from Jan_10_2024_stations
for index, (lat, lon, d_str) in Dec_11_2022_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    wind_speed = wind_speed_2022.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['wind_speed'].item()
    
    # Append wind speed to the list
    wind_speeds.append(wind_speed)

# Add 'wind_speed' column to Jan_10_2024_stations
Dec_11_2022_stations['wind_speed'] = wind_speeds

# Display the updated DataFrame
print(Dec_11_2022_stations)

       LATITUDE   LONGITUDE  STATION_ID  \
3042  46.993439  -65.506909   CAN-NB-11   
3043  47.289956  -68.411945  CAN-NB-134   
3044  47.057010  -67.726300   CAN-NB-38   
3045  46.868300  -68.013600         CAR   
3046  46.868100  -68.012500       CARM1   
...         ...         ...         ...   
4136  44.088600 -103.299700     SD-PN-2   
4137  44.086570 -103.370100    SD-PN-40   
4138  44.497300 -103.871700       SPES2   
4139  44.424500 -103.379300       STMS2   
4140  44.072500 -103.212200       UNRS2   

                                       STATION_TY  STATION_EL  DEM_ELEVAT  \
3042                                     COCORAHS        33.0        28.0   
3043                                     COCORAHS       147.0       152.0   
3044                                     COCORAHS       200.0       180.0   
3045  ARSR4, ASOS, AWIPS, COOPAB, CRS, UA, WSR88D       187.0       184.0   
3046                               COOPAB, SNOCOR       191.0       184.0   
...                  

In [11]:
# Initialize an empty list to store wind speeds
wind_speeds = []

# Extracting data based on lat, lon, and time from Jan_10_2024_stations
for index, (lat, lon, d_str) in Dec_28_2023_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    wind_speed = wind_speed_2023.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['wind_speed'].item()
    
    # Append wind speed to the list
    wind_speeds.append(wind_speed)

# Add 'wind_speed' column to Jan_10_2024_stations
Dec_28_2023_stations['wind_speed'] = wind_speeds

# Display the updated DataFrame
print(Dec_28_2023_stations)


      LATITUDE  LONGITUDE STATION_ID                             STATION_TY  \
1707   43.1653   -95.1467        3SE                                COOPABC   
1708   45.6708   -96.9961        8D3                                   ASOS   
1709   45.4553   -98.4141      ABES2                                COOPABC   
1710   45.4825   -87.8119      ABGW3                                  UCOOP   
1711   45.4558   -98.4128        ABR  ASOS, AWIPS, COOPABC, CRS, UA, WSR88D   
...        ...        ...        ...                                    ...   
3345   47.1472  -108.5913      WRHM8                                   UMRB   
3346   43.5600  -100.7400      WRIS2                                   UMRB   
3347   44.0026  -107.9050      WRMW4                                   UMRB   
3348   43.7322   -98.6939      WTES2                                   UMRB   
3349   46.0135   -99.6876      ZELN8                                   UMRB   

      STATION_EL  DEM_ELEVAT  FOREST_DEN      MD_DE

### Add Min Temp data into attribute dataframe

In [12]:
# Upload 2024 min temp data

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\tmmn_2024.nc"

#read in the file with xarray:
min_temp_2024 = xr.open_dataset(file_path)

min_temp_2024 = min_temp_2024.compute()

min_temp_2024

<xarray.Dataset> Size: 1GB
Dimensions:          (lon: 1386, lat: 585, day: 190, crs: 1)
Coordinates:
  * lon              (lon) float64 11kB -124.8 -124.7 -124.7 ... -67.1 -67.06
  * lat              (lat) float64 5kB 49.4 49.36 49.32 ... 25.15 25.11 25.07
  * day              (day) datetime64[ns] 2kB 2024-01-01 ... 2024-07-08
  * crs              (crs) uint16 2B 3
Data variables:
    air_temperature  (day, lat, lon) float64 1GB nan nan nan nan ... nan nan nan
Attributes: (12/22)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    last_permanent_slice:       130
    last_early_slice:           190
    last_provisional_slice:     184
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [13]:
# Upload 2023 min temp data

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\tmmn_2023.nc"

#read in the file with xarray:
min_temp_2023 = xr.open_dataset(file_path)

min_temp_2023 = min_temp_2023.compute()

min_temp_2023

<xarray.Dataset> Size: 2GB
Dimensions:          (lon: 1386, lat: 585, day: 365, crs: 1)
Coordinates:
  * lon              (lon) float64 11kB -124.8 -124.7 -124.7 ... -67.1 -67.06
  * lat              (lat) float64 5kB 49.4 49.36 49.32 ... 25.15 25.11 25.07
  * day              (day) datetime64[ns] 3kB 2023-01-01 ... 2023-12-31
  * crs              (crs) uint16 2B 3
Data variables:
    air_temperature  (day, lat, lon) float64 2GB nan nan nan nan ... nan nan nan
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       02 March 2024
    note1:                      The projection information for this file is: ...
    note2:                      Citation: Abatzoglou, J.T., 2013, Development...
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [14]:
# Upload 2022 min temp data

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\tmmn_2022.nc"

#read in the file with xarray:
min_temp_2022 = xr.open_dataset(file_path)

min_temp_2022 = min_temp_2022.compute()

min_temp_2022

<xarray.Dataset> Size: 2GB
Dimensions:          (lon: 1386, lat: 585, day: 365, crs: 1)
Coordinates:
  * lon              (lon) float64 11kB -124.8 -124.7 -124.7 ... -67.1 -67.06
  * lat              (lat) float64 5kB 49.4 49.36 49.32 ... 25.15 25.11 25.07
  * day              (day) datetime64[ns] 3kB 2022-01-01 ... 2022-12-31
  * crs              (crs) uint16 2B 3
Data variables:
    air_temperature  (day, lat, lon) float64 2GB nan nan nan nan ... nan nan nan
Attributes: (12/22)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    last_permanent_slice:       305
    last_early_slice:           365
    last_provisional_slice:     359
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [15]:
# Initialize an empty list to store wind speeds
min_temps = []

# Extracting data based on lat, lon, and time from Jan_10_2024_stations
for index, (lat, lon, d_str) in Jan_10_2024_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    air_temps = min_temp_2024.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['air_temperature'].item()
    
    # Append wind speed to the list
    min_temps.append(air_temps)

# Add 'wind_speed' column to Jan_10_2024_stations
Jan_10_2024_stations['min_temp'] = min_temps

# Display the updated DataFrame
print(Jan_10_2024_stations)

      LATITUDE  LONGITUDE        STATION_ID STATION_TY  STATION_EL  \
3401   43.1653   -95.1467               3SE    COOPABC       403.0   
3402   41.4461   -83.3289  41.4461_083.3289    SPOTTER       190.0   
3403   41.6383   -83.5281  41.6383_083.5281    SPOTTER       187.0   
3404   45.6708   -96.9961               8D3       ASOS       353.0   
3405   42.2416   -83.6933             AASM4     COOPAB       254.0   
...        ...        ...               ...        ...         ...   
7053   47.2719  -100.3303             WNGN8       UMRB     -9999.0   
7054   43.3100   -99.6900             WNMS2       UMRB     -9999.0   
7055   43.5600  -100.7400             WRIS2       UMRB     -9999.0   
7056   43.7322   -98.6939             WTES2       UMRB     -9999.0   
7057   46.0135   -99.6876             ZELN8       UMRB     -9999.0   

      DEM_ELEVAT  FOREST_DEN      MD_DEPTH_T  D_SWE_OM    OB_SWE    MD_SWE  \
3401       405.0         3.0   2024-01-10 06  0.009460  0.014676  0.005216   
340

In [16]:
# Initialize an empty list to store wind speeds
min_temps = []

# Extracting data based on lat, lon, and time from Dec_11_2022_stations
for index, (lat, lon, d_str) in Dec_11_2022_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    air_temps = min_temp_2022.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['air_temperature'].item()
    
    # Append wind speed to the list
    min_temps.append(air_temps)

# Add 'wind_speed' column to Dec_11_2022_stations
Dec_11_2022_stations['min_temp'] = min_temps

# Display the updated DataFrame
print(Dec_11_2022_stations)

       LATITUDE   LONGITUDE  STATION_ID  \
3042  46.993439  -65.506909   CAN-NB-11   
3043  47.289956  -68.411945  CAN-NB-134   
3044  47.057010  -67.726300   CAN-NB-38   
3045  46.868300  -68.013600         CAR   
3046  46.868100  -68.012500       CARM1   
...         ...         ...         ...   
4136  44.088600 -103.299700     SD-PN-2   
4137  44.086570 -103.370100    SD-PN-40   
4138  44.497300 -103.871700       SPES2   
4139  44.424500 -103.379300       STMS2   
4140  44.072500 -103.212200       UNRS2   

                                       STATION_TY  STATION_EL  DEM_ELEVAT  \
3042                                     COCORAHS        33.0        28.0   
3043                                     COCORAHS       147.0       152.0   
3044                                     COCORAHS       200.0       180.0   
3045  ARSR4, ASOS, AWIPS, COOPAB, CRS, UA, WSR88D       187.0       184.0   
3046                               COOPAB, SNOCOR       191.0       184.0   
...                  

In [17]:
# Initialize an empty list to store wind speeds
min_temps = []

# Extracting data based on lat, lon, and time from Dec_28_2023_stations
for index, (lat, lon, d_str) in Dec_28_2023_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    air_temps = min_temp_2023.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['air_temperature'].item()
    
    # Append wind speed to the list
    min_temps.append(air_temps)

# Add 'wind_speed' column to Dec_28_2023_stations
Dec_28_2023_stations['min_temp'] = min_temps

# Display the updated DataFrame
print(Dec_28_2023_stations)

      LATITUDE  LONGITUDE STATION_ID                             STATION_TY  \
1707   43.1653   -95.1467        3SE                                COOPABC   
1708   45.6708   -96.9961        8D3                                   ASOS   
1709   45.4553   -98.4141      ABES2                                COOPABC   
1710   45.4825   -87.8119      ABGW3                                  UCOOP   
1711   45.4558   -98.4128        ABR  ASOS, AWIPS, COOPABC, CRS, UA, WSR88D   
...        ...        ...        ...                                    ...   
3345   47.1472  -108.5913      WRHM8                                   UMRB   
3346   43.5600  -100.7400      WRIS2                                   UMRB   
3347   44.0026  -107.9050      WRMW4                                   UMRB   
3348   43.7322   -98.6939      WTES2                                   UMRB   
3349   46.0135   -99.6876      ZELN8                                   UMRB   

      STATION_EL  DEM_ELEVAT  FOREST_DEN      MD_DE

### Add Max Temp into attribute dataframe

In [18]:
# Upload 2024 max temp data

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\tmmx_2024.nc"

#read in the file with xarray:
max_temp_2024 = xr.open_dataset(file_path)

max_temp_2024 = max_temp_2024.compute()

max_temp_2024

<xarray.Dataset> Size: 1GB
Dimensions:          (lon: 1386, lat: 585, day: 190, crs: 1)
Coordinates:
  * lon              (lon) float64 11kB -124.8 -124.7 -124.7 ... -67.1 -67.06
  * lat              (lat) float64 5kB 49.4 49.36 49.32 ... 25.15 25.11 25.07
  * day              (day) datetime64[ns] 2kB 2024-01-01 ... 2024-07-08
  * crs              (crs) uint16 2B 3
Data variables:
    air_temperature  (day, lat, lon) float64 1GB nan nan nan nan ... nan nan nan
Attributes: (12/22)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    last_permanent_slice:       130
    last_early_slice:           190
    last_provisional_slice:     184
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [19]:
# Upload 2023 max temp data

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\tmmx_2023.nc"

#read in the file with xarray:
max_temp_2023 = xr.open_dataset(file_path)

max_temp_2023 = max_temp_2023.compute()

max_temp_2023

<xarray.Dataset> Size: 2GB
Dimensions:          (lon: 1386, lat: 585, day: 365, crs: 1)
Coordinates:
  * lon              (lon) float64 11kB -124.8 -124.7 -124.7 ... -67.1 -67.06
  * lat              (lat) float64 5kB 49.4 49.36 49.32 ... 25.15 25.11 25.07
  * day              (day) datetime64[ns] 3kB 2023-01-01 ... 2023-12-31
  * crs              (crs) uint16 2B 3
Data variables:
    air_temperature  (day, lat, lon) float64 2GB nan nan nan nan ... nan nan nan
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       02 March 2024
    note1:                      The projection information for this file is: ...
    note2:                      Citation: Abatzoglou, J.T., 2013, Development...
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [20]:
# Upload 2024 max temp data

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\tmmx_2022.nc"

#read in the file with xarray:
max_temp_2022 = xr.open_dataset(file_path)

max_temp_2022 = max_temp_2022.compute()

max_temp_2022

<xarray.Dataset> Size: 2GB
Dimensions:          (lon: 1386, lat: 585, day: 365, crs: 1)
Coordinates:
  * lon              (lon) float64 11kB -124.8 -124.7 -124.7 ... -67.1 -67.06
  * lat              (lat) float64 5kB 49.4 49.36 49.32 ... 25.15 25.11 25.07
  * day              (day) datetime64[ns] 3kB 2022-01-01 ... 2022-12-31
  * crs              (crs) uint16 2B 3
Data variables:
    air_temperature  (day, lat, lon) float64 2GB nan nan nan nan ... nan nan nan
Attributes: (12/22)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    last_permanent_slice:       305
    last_early_slice:           365
    last_provisional_slice:     359
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [21]:
# Initialize an empty list to store wind speeds
max_temps = []

# Extracting data based on lat, lon, and time from Jan_10_2024_stations
for index, (lat, lon, d_str) in Jan_10_2024_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    air_temps = max_temp_2024.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['air_temperature'].item()
    
    # Append wind speed to the list
    max_temps.append(air_temps)

# Add 'wind_speed' column to Jan_10_2024_stations
Jan_10_2024_stations['max_temp'] = max_temps

# Display the updated DataFrame
print(Jan_10_2024_stations)

      LATITUDE  LONGITUDE        STATION_ID STATION_TY  STATION_EL  \
3401   43.1653   -95.1467               3SE    COOPABC       403.0   
3402   41.4461   -83.3289  41.4461_083.3289    SPOTTER       190.0   
3403   41.6383   -83.5281  41.6383_083.5281    SPOTTER       187.0   
3404   45.6708   -96.9961               8D3       ASOS       353.0   
3405   42.2416   -83.6933             AASM4     COOPAB       254.0   
...        ...        ...               ...        ...         ...   
7053   47.2719  -100.3303             WNGN8       UMRB     -9999.0   
7054   43.3100   -99.6900             WNMS2       UMRB     -9999.0   
7055   43.5600  -100.7400             WRIS2       UMRB     -9999.0   
7056   43.7322   -98.6939             WTES2       UMRB     -9999.0   
7057   46.0135   -99.6876             ZELN8       UMRB     -9999.0   

      DEM_ELEVAT  FOREST_DEN      MD_DEPTH_T  D_SWE_OM    OB_SWE    MD_SWE  \
3401       405.0         3.0   2024-01-10 06  0.009460  0.014676  0.005216   
340

In [22]:
# Initialize an empty list to store wind speeds
max_temps = []

# Extracting data based on lat, lon, and time from Dec_11_2022_station
for index, (lat, lon, d_str) in Dec_11_2022_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    air_temps = max_temp_2022.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['air_temperature'].item()
    
    # Append wind speed to the list
    max_temps.append(air_temps)

# Add 'wind_speed' column to Dec_11_2022_station
Dec_11_2022_stations['max_temp'] = max_temps

# Display the updated DataFrame
print(Dec_11_2022_stations)

       LATITUDE   LONGITUDE  STATION_ID  \
3042  46.993439  -65.506909   CAN-NB-11   
3043  47.289956  -68.411945  CAN-NB-134   
3044  47.057010  -67.726300   CAN-NB-38   
3045  46.868300  -68.013600         CAR   
3046  46.868100  -68.012500       CARM1   
...         ...         ...         ...   
4136  44.088600 -103.299700     SD-PN-2   
4137  44.086570 -103.370100    SD-PN-40   
4138  44.497300 -103.871700       SPES2   
4139  44.424500 -103.379300       STMS2   
4140  44.072500 -103.212200       UNRS2   

                                       STATION_TY  STATION_EL  DEM_ELEVAT  \
3042                                     COCORAHS        33.0        28.0   
3043                                     COCORAHS       147.0       152.0   
3044                                     COCORAHS       200.0       180.0   
3045  ARSR4, ASOS, AWIPS, COOPAB, CRS, UA, WSR88D       187.0       184.0   
3046                               COOPAB, SNOCOR       191.0       184.0   
...                  

In [23]:
# Initialize an empty list to store wind speeds
max_temps = []

# Extracting data based on lat, lon, and time from Dec_28_2023_stations
for index, (lat, lon, d_str) in Dec_28_2023_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    air_temps = max_temp_2023.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['air_temperature'].item()
    
    # Append wind speed to the list
    max_temps.append(air_temps)

# Add 'wind_speed' column to Dec_11_2022_station
Dec_28_2023_stations['max_temp'] = max_temps

# Display the updated DataFrame
print(Dec_28_2023_stations)

      LATITUDE  LONGITUDE STATION_ID                             STATION_TY  \
1707   43.1653   -95.1467        3SE                                COOPABC   
1708   45.6708   -96.9961        8D3                                   ASOS   
1709   45.4553   -98.4141      ABES2                                COOPABC   
1710   45.4825   -87.8119      ABGW3                                  UCOOP   
1711   45.4558   -98.4128        ABR  ASOS, AWIPS, COOPABC, CRS, UA, WSR88D   
...        ...        ...        ...                                    ...   
3345   47.1472  -108.5913      WRHM8                                   UMRB   
3346   43.5600  -100.7400      WRIS2                                   UMRB   
3347   44.0026  -107.9050      WRMW4                                   UMRB   
3348   43.7322   -98.6939      WTES2                                   UMRB   
3349   46.0135   -99.6876      ZELN8                                   UMRB   

      STATION_EL  DEM_ELEVAT  FOREST_DEN      MD_DE

### Add Surface Radiation into attribute dataframe

In [24]:
# Upload 2024 surface radiation

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\srad_2024.nc"

#read in the file with xarray:
surf_rad_2024 = xr.open_dataset(file_path)

surf_rad_2024 = surf_rad_2024.compute()

surf_rad_2024

<xarray.Dataset> Size: 1GB
Dimensions:                                    (lon: 1386, lat: 585, day: 190,
                                                crs: 1)
Coordinates:
  * lon                                        (lon) float64 11kB -124.8 ... ...
  * lat                                        (lat) float64 5kB 49.4 ... 25.07
  * day                                        (day) datetime64[ns] 2kB 2024-...
  * crs                                        (crs) uint16 2B 3
Data variables:
    surface_downwelling_shortwave_flux_in_air  (day, lat, lon) float64 1GB na...
Attributes: (12/22)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    last_permanent_slice:       130
    last_early_slice:           190
    last_provisional_slice:     184
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [25]:
# Upload 2023 surface radiation

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\srad_2023.nc"

#read in the file with xarray:
surf_rad_2023 = xr.open_dataset(file_path)

surf_rad_2023 = surf_rad_2023.compute()

surf_rad_2023

<xarray.Dataset> Size: 2GB
Dimensions:                                    (lon: 1386, lat: 585, day: 365,
                                                crs: 1)
Coordinates:
  * lon                                        (lon) float64 11kB -124.8 ... ...
  * lat                                        (lat) float64 5kB 49.4 ... 25.07
  * day                                        (day) datetime64[ns] 3kB 2023-...
  * crs                                        (crs) uint16 2B 3
Data variables:
    surface_downwelling_shortwave_flux_in_air  (day, lat, lon) float64 2GB na...
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       02 March 2024
    note1:                      The projection information for this file is: ...
    note2:                      Citation: Abatzoglou, J.T., 2013, Development...
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [26]:
# Upload 2022 surface radiation

# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\srad_2022.nc"

#read in the file with xarray:
surf_rad_2022 = xr.open_dataset(file_path)

surf_rad_2022 = surf_rad_2022.compute()

surf_rad_2022

<xarray.Dataset> Size: 2GB
Dimensions:                                    (lon: 1386, lat: 585, day: 365,
                                                crs: 1)
Coordinates:
  * lon                                        (lon) float64 11kB -124.8 ... ...
  * lat                                        (lat) float64 5kB 49.4 ... 25.07
  * day                                        (day) datetime64[ns] 3kB 2022-...
  * crs                                        (crs) uint16 2B 3
Data variables:
    surface_downwelling_shortwave_flux_in_air  (day, lat, lon) float64 2GB na...
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       15 January 2023
    note1:                      The projection information for this file is: ...
    note2:                      Citation: Abatzoglou, J.T., 2013, Development...
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [27]:
# Initialize an empty list to store surface radiation
surf_rad = []

# Extracting data based on lat, lon, and time from Jan_10_2024_stations
for index, (lat, lon, d_str) in Jan_10_2024_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    rad_levels = surf_rad_2024.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['surface_downwelling_shortwave_flux_in_air'].item()
    
    # Append wind speed to the list
    surf_rad.append(rad_levels)

# Add 'wind_speed' column to Jan_10_2024_stations
Jan_10_2024_stations['surf_radiation'] = surf_rad

# Display the updated DataFrame
print(Jan_10_2024_stations)

      LATITUDE  LONGITUDE        STATION_ID STATION_TY  STATION_EL  \
3401   43.1653   -95.1467               3SE    COOPABC       403.0   
3402   41.4461   -83.3289  41.4461_083.3289    SPOTTER       190.0   
3403   41.6383   -83.5281  41.6383_083.5281    SPOTTER       187.0   
3404   45.6708   -96.9961               8D3       ASOS       353.0   
3405   42.2416   -83.6933             AASM4     COOPAB       254.0   
...        ...        ...               ...        ...         ...   
7053   47.2719  -100.3303             WNGN8       UMRB     -9999.0   
7054   43.3100   -99.6900             WNMS2       UMRB     -9999.0   
7055   43.5600  -100.7400             WRIS2       UMRB     -9999.0   
7056   43.7322   -98.6939             WTES2       UMRB     -9999.0   
7057   46.0135   -99.6876             ZELN8       UMRB     -9999.0   

      DEM_ELEVAT  FOREST_DEN      MD_DEPTH_T  D_SWE_OM    OB_SWE    MD_SWE  \
3401       405.0         3.0   2024-01-10 06  0.009460  0.014676  0.005216   
340

In [28]:
# Initialize an empty list to store surface radiation
surf_rad = []

# Extracting data based on lat, lon, and time from Dec_11_2022_stations
for index, (lat, lon, d_str) in Dec_11_2022_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    rad_levels = surf_rad_2022.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['surface_downwelling_shortwave_flux_in_air'].item()
    
    # Append wind speed to the list
    surf_rad.append(rad_levels)

# Add 'wind_speed' column to Dec_11_2022_stations
Dec_11_2022_stations['surf_radiation'] = surf_rad

# Display the updated DataFrame
print(Dec_11_2022_stations)

       LATITUDE   LONGITUDE  STATION_ID  \
3042  46.993439  -65.506909   CAN-NB-11   
3043  47.289956  -68.411945  CAN-NB-134   
3044  47.057010  -67.726300   CAN-NB-38   
3045  46.868300  -68.013600         CAR   
3046  46.868100  -68.012500       CARM1   
...         ...         ...         ...   
4136  44.088600 -103.299700     SD-PN-2   
4137  44.086570 -103.370100    SD-PN-40   
4138  44.497300 -103.871700       SPES2   
4139  44.424500 -103.379300       STMS2   
4140  44.072500 -103.212200       UNRS2   

                                       STATION_TY  STATION_EL  DEM_ELEVAT  \
3042                                     COCORAHS        33.0        28.0   
3043                                     COCORAHS       147.0       152.0   
3044                                     COCORAHS       200.0       180.0   
3045  ARSR4, ASOS, AWIPS, COOPAB, CRS, UA, WSR88D       187.0       184.0   
3046                               COOPAB, SNOCOR       191.0       184.0   
...                  

In [29]:
# Initialize an empty list to store surface radiation
surf_rad = []

# Extracting data based on lat, lon, and time from Dec_28_2023_stations
for index, (lat, lon, d_str) in Dec_28_2023_stations[['LATITUDE', 'LONGITUDE', 'MD_DEPTH_T']].iterrows():
    # Clean up unexpected characters (if any) in d_str and convert to datetime object
    d_str_cleaned = d_str.replace(':', '')  # Remove unexpected character ':'

    rad_levels = surf_rad_2023.sel(lat=lat, lon=lon, day=d_str_cleaned, method='nearest')['surface_downwelling_shortwave_flux_in_air'].item()
    
    # Append wind speed to the list
    surf_rad.append(rad_levels)

# Add 'wind_speed' column to Dec_28_2023_stations
Dec_28_2023_stations['surf_radiation'] = surf_rad

# Display the updated DataFrame
print(Dec_28_2023_stations)

      LATITUDE  LONGITUDE STATION_ID                             STATION_TY  \
1707   43.1653   -95.1467        3SE                                COOPABC   
1708   45.6708   -96.9961        8D3                                   ASOS   
1709   45.4553   -98.4141      ABES2                                COOPABC   
1710   45.4825   -87.8119      ABGW3                                  UCOOP   
1711   45.4558   -98.4128        ABR  ASOS, AWIPS, COOPABC, CRS, UA, WSR88D   
...        ...        ...        ...                                    ...   
3345   47.1472  -108.5913      WRHM8                                   UMRB   
3346   43.5600  -100.7400      WRIS2                                   UMRB   
3347   44.0026  -107.9050      WRMW4                                   UMRB   
3348   43.7322   -98.6939      WTES2                                   UMRB   
3349   46.0135   -99.6876      ZELN8                                   UMRB   

      STATION_EL  DEM_ELEVAT  FOREST_DEN      MD_DE

### Add cumulative precipitation data into attribute dataframe

In [39]:
# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\pr_2024.nc"

# Open the dataset
precip_2024 = xr.open_dataset(file_path)

# Select a subset of the data
precip_2024 = precip_2024.sel(day=slice('2024-01-01', '2024-01-30'))

# Compute the subset
precip_2024 = precip_2024.compute()

precip_2024

<xarray.Dataset> Size: 195MB
Dimensions:               (lon: 1386, lat: 585, day: 30, crs: 1)
Coordinates:
  * lon                   (lon) float64 11kB -124.8 -124.7 ... -67.1 -67.06
  * lat                   (lat) float64 5kB 49.4 49.36 49.32 ... 25.11 25.07
  * day                   (day) datetime64[ns] 240B 2024-01-01 ... 2024-01-30
  * crs                   (crs) uint16 2B 3
Data variables:
    precipitation_amount  (day, lat, lon) float64 195MB nan nan nan ... nan nan
Attributes: (12/22)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    last_permanent_slice:       130
    last_early_slice:           190
    last_provisional_slice:     184
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [40]:
# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\pr_2023.nc"

# Open the dataset
precip_2023 = xr.open_dataset(file_path)

# Select a subset of the data
precip_2023 = precip_2023.sel(day=slice('2023-10-01', '2023-12-31'))

# Compute the subset
precip_2023 = precip_2023.compute()

precip_2023

<xarray.Dataset> Size: 597MB
Dimensions:               (lon: 1386, lat: 585, day: 92, crs: 1)
Coordinates:
  * lon                   (lon) float64 11kB -124.8 -124.7 ... -67.1 -67.06
  * lat                   (lat) float64 5kB 49.4 49.36 49.32 ... 25.11 25.07
  * day                   (day) datetime64[ns] 736B 2023-10-01 ... 2023-12-31
  * crs                   (crs) uint16 2B 3
Data variables:
    precipitation_amount  (day, lat, lon) float64 597MB nan nan nan ... nan nan
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       02 March 2024
    note1:                      The projection information for this file is: ...
    note2:                      Citation: Abatzoglou, J.T., 2013, Development...
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [41]:
# Path to your NetCDF file
file_path = r"C:\Users\clemasters\Research Triangle Institute\CIROH Basecamp - Documents\Projects\0218723.014 - SNODAS\Data\station_attribute_data\pr_2022.nc"

# Open the dataset
precip_2022 = xr.open_dataset(file_path)

# Select a subset of the data
precip_2022 = precip_2022.sel(day=slice('2022-10-01', '2022-12-31'))

# Compute the subset
precip_2022 = precip_2022.compute()

precip_2022

<xarray.Dataset> Size: 597MB
Dimensions:               (lon: 1386, lat: 585, day: 92, crs: 1)
Coordinates:
  * lon                   (lon) float64 11kB -124.8 -124.7 ... -67.1 -67.06
  * lat                   (lat) float64 5kB 49.4 49.36 49.32 ... 25.11 25.07
  * day                   (day) datetime64[ns] 736B 2022-10-01 ... 2022-12-31
  * crs                   (crs) uint16 2B 3
Data variables:
    precipitation_amount  (day, lat, lon) float64 597MB nan nan nan ... nan nan
Attributes: (12/22)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    last_permanent_slice:       305
    last_early_slice:           365
    last_provisional_slice:     359
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [42]:
# Merge datasets along the time dimension
merged_precip = xr.concat([precip_2022, precip_2023, precip_2024], dim='day')

# Optionally, you may want to sort the dataset by the 'day' dimension
merged_precip = merged_precip.sortby('day')

merged_precip

<xarray.Dataset> Size: 1GB
Dimensions:               (lon: 1386, lat: 585, day: 214, crs: 1)
Coordinates:
  * lon                   (lon) float64 11kB -124.8 -124.7 ... -67.1 -67.06
  * lat                   (lat) float64 5kB 49.4 49.36 49.32 ... 25.11 25.07
  * day                   (day) datetime64[ns] 2kB 2022-10-01 ... 2024-01-30
  * crs                   (crs) uint16 2B 3
Data variables:
    precipitation_amount  (day, lat, lon) float64 1GB nan nan nan ... nan nan
Attributes: (12/22)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    last_permanent_slice:       305
    last_early_slice:           365
    last_provisional_slice:     359
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...

In [55]:
precip_Dec_2022  = merged_precip.sel(day=slice('2022-10-01', '2022-12-11'))

#calcualte the total precipitation across all timesteps at every grid point
total_precip_Dec_2022 = precip_Dec_2022.sum(dim = 'day')

total_precip_Dec_2022

<xarray.Dataset> Size: 7MB
Dimensions:               (lon: 1386, lat: 585, crs: 1)
Coordinates:
  * lon                   (lon) float64 11kB -124.8 -124.7 ... -67.1 -67.06
  * lat                   (lat) float64 5kB 49.4 49.36 49.32 ... 25.11 25.07
  * crs                   (crs) uint16 2B 3
Data variables:
    precipitation_amount  (lat, lon) float64 6MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0

In [56]:
precip_Dec_2023  = merged_precip.sel(day=slice('2023-10-01', '2023-12-28'))

#calcualte the total precipitation across all timesteps at every grid point
total_precip_Dec_2023 = precip_Dec_2023.sum(dim = 'day')

total_precip_Dec_2023

<xarray.Dataset> Size: 7MB
Dimensions:               (lon: 1386, lat: 585, crs: 1)
Coordinates:
  * lon                   (lon) float64 11kB -124.8 -124.7 ... -67.1 -67.06
  * lat                   (lat) float64 5kB 49.4 49.36 49.32 ... 25.11 25.07
  * crs                   (crs) uint16 2B 3
Data variables:
    precipitation_amount  (lat, lon) float64 6MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0

In [45]:
precip_Jan_2024  = merged_precip.sel(day=slice('2023-10-01', '2024-01-10'))

#calcualte the total precipitation across all timesteps at every grid point
total_precip_Jan_2024 = precip_Jan_2024.sum(dim = 'day')

total_precip_Jan_2024

<xarray.Dataset> Size: 7MB
Dimensions:               (lon: 1386, lat: 585, crs: 1)
Coordinates:
  * lon                   (lon) float64 11kB -124.8 -124.7 ... -67.1 -67.06
  * lat                   (lat) float64 5kB 49.4 49.36 49.32 ... 25.11 25.07
  * crs                   (crs) uint16 2B 3
Data variables:
    precipitation_amount  (lat, lon) float64 6MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0

In [53]:
# Initialize an empty list to store surface radiation
cummulative_precip = []

# Extracting data based on lat, lon, and time from Dec_28_2023_stations
for index, (lat, lon) in Dec_11_2022_stations[['LATITUDE', 'LONGITUDE']].iterrows():

    precip_levels = total_precip_Dec_2022.sel(lat=lat, lon=lon, method='nearest')['precipitation_amount'].item()
    
    # Append wind speed to the list
    cummulative_precip.append(precip_levels)

# Add 'wind_speed' column to Dec_11_2022_stations
Dec_11_2022_stations['cummulative_precip'] = cummulative_precip

# Display the updated DataFrame
print(Dec_11_2022_stations)

       LATITUDE   LONGITUDE  STATION_ID  \
3042  46.993439  -65.506909   CAN-NB-11   
3043  47.289956  -68.411945  CAN-NB-134   
3044  47.057010  -67.726300   CAN-NB-38   
3045  46.868300  -68.013600         CAR   
3046  46.868100  -68.012500       CARM1   
...         ...         ...         ...   
4136  44.088600 -103.299700     SD-PN-2   
4137  44.086570 -103.370100    SD-PN-40   
4138  44.497300 -103.871700       SPES2   
4139  44.424500 -103.379300       STMS2   
4140  44.072500 -103.212200       UNRS2   

                                       STATION_TY  STATION_EL  DEM_ELEVAT  \
3042                                     COCORAHS        33.0        28.0   
3043                                     COCORAHS       147.0       152.0   
3044                                     COCORAHS       200.0       180.0   
3045  ARSR4, ASOS, AWIPS, COOPAB, CRS, UA, WSR88D       187.0       184.0   
3046                               COOPAB, SNOCOR       191.0       184.0   
...                  

In [54]:
# Initialize an empty list to store surface radiation
cummulative_precip = []

# Extracting data based on lat, lon, and time from Jan_10_2024_stations
for index, (lat, lon) in Jan_10_2024_stations[['LATITUDE', 'LONGITUDE']].iterrows():

    precip_levels = total_precip_Jan_2024.sel(lat=lat, lon=lon, method='nearest')['precipitation_amount'].item()
    
    # Append wind speed to the list
    cummulative_precip.append(precip_levels)

# Add 'wind_speed' column to Jan_10_2024_stations
Jan_10_2024_stations['cummulative_precip'] = cummulative_precip

# Display the updated DataFrame
print(Jan_10_2024_stations)

      LATITUDE  LONGITUDE        STATION_ID STATION_TY  STATION_EL  \
3401   43.1653   -95.1467               3SE    COOPABC       403.0   
3402   41.4461   -83.3289  41.4461_083.3289    SPOTTER       190.0   
3403   41.6383   -83.5281  41.6383_083.5281    SPOTTER       187.0   
3404   45.6708   -96.9961               8D3       ASOS       353.0   
3405   42.2416   -83.6933             AASM4     COOPAB       254.0   
...        ...        ...               ...        ...         ...   
7053   47.2719  -100.3303             WNGN8       UMRB     -9999.0   
7054   43.3100   -99.6900             WNMS2       UMRB     -9999.0   
7055   43.5600  -100.7400             WRIS2       UMRB     -9999.0   
7056   43.7322   -98.6939             WTES2       UMRB     -9999.0   
7057   46.0135   -99.6876             ZELN8       UMRB     -9999.0   

      DEM_ELEVAT  FOREST_DEN      MD_DEPTH_T  D_SWE_OM    OB_SWE    MD_SWE  \
3401       405.0         3.0   2024-01-10 06  0.009460  0.014676  0.005216   
340

In [57]:
# Initialize an empty list to store surface radiation
cummulative_precip = []

# Extracting data based on lat, lon, and time from Dec_28_2023_stations
for index, (lat, lon) in Dec_28_2023_stations[['LATITUDE', 'LONGITUDE']].iterrows():

    precip_levels = total_precip_Dec_2023.sel(lat=lat, lon=lon, method='nearest')['precipitation_amount'].item()
    
    # Append wind speed to the list
    cummulative_precip.append(precip_levels)

# Add 'wind_speed' column to Dec_28_2023_stations
Dec_28_2023_stations['cummulative_precip'] = cummulative_precip

# Display the updated DataFrame
print(Dec_28_2023_stations)

      LATITUDE  LONGITUDE STATION_ID                             STATION_TY  \
1707   43.1653   -95.1467        3SE                                COOPABC   
1708   45.6708   -96.9961        8D3                                   ASOS   
1709   45.4553   -98.4141      ABES2                                COOPABC   
1710   45.4825   -87.8119      ABGW3                                  UCOOP   
1711   45.4558   -98.4128        ABR  ASOS, AWIPS, COOPABC, CRS, UA, WSR88D   
...        ...        ...        ...                                    ...   
3345   47.1472  -108.5913      WRHM8                                   UMRB   
3346   43.5600  -100.7400      WRIS2                                   UMRB   
3347   44.0026  -107.9050      WRMW4                                   UMRB   
3348   43.7322   -98.6939      WTES2                                   UMRB   
3349   46.0135   -99.6876      ZELN8                                   UMRB   

      STATION_EL  DEM_ELEVAT  FOREST_DEN      MD_DE

### Add slope and aspect to attribute dataframe from DEM layer

### Add elevation to attribute dataframe

In [58]:
# Select 'STATION_ID' and 'STATION_EL' columns from combined_stations and umrb_station_attributes
STATION_EL = Dec_11_2022_stations[['STATION_ID', 'STATION_EL']]

STATION_EL_UMRB = umrb_station_attributes[['STATION_ID', 'STATION_EL']]

# Combine 'STATION_EL' data from both DataFrames
STATION_EL_combined = pd.concat([STATION_EL, STATION_EL_UMRB])

# Filter rows where 'STATION_ID' exists in combined_stations['STATION_ID']
STATION_EL_combined = STATION_EL_combined[STATION_EL_combined['STATION_ID'].isin(Dec_11_2022_stations['STATION_ID'])]

Dec_11_2022_stations = Dec_11_2022_stations.drop(columns=['STATION_EL'])

# Merge STATION_EL_combined with combined_stations based on STATION_ID
Dec_11_2022_stations = Dec_11_2022_stations.merge(STATION_EL_combined, on='STATION_ID', suffixes=('_old', '_new'))

# Remove duplicate rows
Dec_11_2022_stations = Dec_11_2022_stations.drop_duplicates()

# Filter out rows where STATION_EL is not equal to -9999.0
Dec_11_2022_stations = Dec_11_2022_stations[Dec_11_2022_stations['STATION_EL'] != -9999.0]

Dec_11_2022_stations

,LATITUDE,LONGITUDE,STATION_ID,STATION_TY,DEM_ELEVAT,FOREST_DEN,MD_DEPTH_T,D_SWE_OM,OB_SWE,MD_SWE,wind_speed,min_temp,max_temp,surf_radiation,cummulative_precip,STATION_EL
0,46.993439,-65.506909,CAN-NB-11,COCORAHS,28.0,54.0,2022-12-11 09,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.0,33.0
1,47.289956,-68.411945,CAN-NB-134,COCORAHS,152.0,24.0,2022-12-11 12,-0.016242,0.000000,0.016242,3.3,259.5,266.6,61.5,261.6,147.0
2,47.057010,-67.726300,CAN-NB-38,COCORAHS,180.0,18.0,2022-12-11 12,-0.012468,0.000000,0.012468,NaN,NaN,NaN,NaN,0.0,200.0
3,46.868300,-68.013600,CAR,"ARSR4, ASOS, AWIPS, COOPAB, CRS, UA, WSR88D",184.0,14.0,2022-12-11 12,0.000000,0.000000,0.000000,3.3,259.5,267.4,68.6,287.3,187.0
4,46.868100,-68.012500,CARM1,"COOPAB, SNOCOR",184.0,14.0,2022-12-11 12,0.000000,0.000000,0.000000,3.3,259.5,267.4,68.6,287.3,191.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1094,44.088600,-103.299700,SD-PN-2,COCORAHS,1052.0,17.0,2022-12-11 14,0.000000,0.000000,0.000000,4.4,262.9,276.5,69.3,18.8,1056.0
1095,44.086570,-103.370100,SD-PN-40,COCORAHS,1284.0,30.0,2022-12-11 14,0.000000,0.000000,0.000000,4.4,263.4,276.8,69.8,21.7,1282.0
1096,44.497300,-103.871700,SPES2,COOPABC,1108.0,9.0,2022-12-10 16,0.000000,0.000000,0.000000,5.1,268.3,287.0,76.3,39.2,1108.0
1097,44.424500,-103.379300,STMS2,MESO-ST,985.0,1.0,2022-12-10 20,0.000102,0.000102,0.000000,3.3,268.4,286.9,74.2,30.1,987.0


In [59]:
# Select 'STATION_ID' and 'STATION_EL' columns from combined_stations and umrb_station_attributes
STATION_EL = Dec_28_2023_stations[['STATION_ID', 'STATION_EL']]

STATION_EL_UMRB = umrb_station_attributes[['STATION_ID', 'STATION_EL']]

# Combine 'STATION_EL' data from both DataFrames
STATION_EL_combined = pd.concat([STATION_EL, STATION_EL_UMRB])

# Filter rows where 'STATION_ID' exists in combined_stations['STATION_ID']
STATION_EL_combined = STATION_EL_combined[STATION_EL_combined['STATION_ID'].isin(Dec_28_2023_stations['STATION_ID'])]

Dec_28_2023_stations = Dec_28_2023_stations.drop(columns=['STATION_EL'])

# Merge STATION_EL_combined with combined_stations based on STATION_ID
Dec_28_2023_stations = Dec_28_2023_stations.merge(STATION_EL_combined, on='STATION_ID', suffixes=('_old', '_new'))

# Remove duplicate rows
Dec_28_2023_stations = Dec_28_2023_stations.drop_duplicates()

# Filter out rows where STATION_EL is not equal to -9999.0
Dec_28_2023_stations = Dec_28_2023_stations[Dec_28_2023_stations['STATION_EL'] != -9999.0]

Dec_28_2023_stations

,LATITUDE,LONGITUDE,STATION_ID,STATION_TY,DEM_ELEVAT,FOREST_DEN,MD_DEPTH_T,D_SWE_OM,OB_SWE,MD_SWE,wind_speed,min_temp,max_temp,surf_radiation,cummulative_precip,STATION_EL
0,43.1653,-95.1467,3SE,COOPABC,405.0,3.0,2023-12-28 06,0.000000,0.000000,0.000000,3.9,267.3,277.3,73.4,220.7,403.0
1,45.6708,-96.9961,8D3,ASOS,350.0,5.0,2023-12-28 12,-0.020945,0.013836,0.034780,4.3,266.8,277.8,74.0,166.0,353.0
2,45.4553,-98.4141,ABES2,COOPABC,395.0,4.0,2023-12-28 12,-0.021128,0.014524,0.035652,3.3,264.0,276.9,73.7,92.4,395.0
3,45.4825,-87.8119,ABGW3,UCOOP,223.0,53.0,2023-12-28 12,0.000000,0.000000,0.000000,3.5,266.0,279.3,78.1,179.4,224.0
4,45.4558,-98.4128,ABR,"ASOS, AWIPS, COOPABC, CRS, UA, WSR88D",395.0,4.0,2023-12-28 12,-0.021128,0.014524,0.035652,3.3,264.0,276.9,73.7,92.4,396.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1516,40.2094,-102.8117,YMAC2,COOPB,1274.0,0.0,2023-12-27 14,0.001358,0.009291,0.007933,3.6,266.6,276.6,99.1,40.3,1279.0
1517,40.9156,-97.5985,YORK005,NERAIN,493.0,2.0,2023-12-28 13,-0.001748,0.011208,0.012956,4.1,265.1,275.1,94.3,110.0,494.0
1518,40.9152,-97.5997,YORN1,COOPAB,493.0,2.0,2023-12-27 15,0.000146,0.013222,0.013076,3.7,263.6,273.4,92.5,110.0,495.0
1519,42.8783,-97.3633,YTNS2,COOPABC,358.0,5.0,2023-12-28 15,-0.022372,0.000000,0.022372,3.8,264.6,274.8,86.6,146.2,357.0


In [60]:
# Select 'STATION_ID' and 'STATION_EL' columns from combined_stations and umrb_station_attributes
STATION_EL = Jan_10_2024_stations[['STATION_ID', 'STATION_EL']]

STATION_EL_UMRB = umrb_station_attributes[['STATION_ID', 'STATION_EL']]

# Combine 'STATION_EL' data from both DataFrames
STATION_EL_combined = pd.concat([STATION_EL, STATION_EL_UMRB])

# Filter rows where 'STATION_ID' exists in combined_stations['STATION_ID']
STATION_EL_combined = STATION_EL_combined[STATION_EL_combined['STATION_ID'].isin(Jan_10_2024_stations['STATION_ID'])]

Jan_10_2024_stations = Jan_10_2024_stations.drop(columns=['STATION_EL'])

# Merge STATION_EL_combined with combined_stations based on STATION_ID
Jan_10_2024_stations = Jan_10_2024_stations.merge(STATION_EL_combined, on='STATION_ID', suffixes=('_old', '_new'))

# Remove duplicate rows
Jan_10_2024_stations = Jan_10_2024_stations.drop_duplicates()

# Filter out rows where STATION_EL is not equal to -9999.0
Jan_10_2024_stations = Jan_10_2024_stations[Jan_10_2024_stations['STATION_EL'] != -9999.0]

Jan_10_2024_stations

,LATITUDE,LONGITUDE,STATION_ID,STATION_TY,DEM_ELEVAT,FOREST_DEN,MD_DEPTH_T,D_SWE_OM,OB_SWE,MD_SWE,wind_speed,min_temp,max_temp,surf_radiation,cummulative_precip,STATION_EL
0,43.1653,-95.1467,3SE,COOPABC,405.0,3.0,2024-01-10 06,0.009460,0.014676,0.005216,2.4,258.3,267.1,76.7,230.9,403.0
1,41.4461,-83.3289,41.4461_083.3289,SPOTTER,191.0,4.0,2024-01-09 13,-0.009658,0.000006,0.009664,7.8,272.7,276.1,47.7,191.6,190.0
2,41.6383,-83.5281,41.6383_083.5281,SPOTTER,182.0,6.0,2024-01-09 13,-0.005330,0.001103,0.006434,7.8,273.1,275.9,49.6,160.6,187.0
3,45.6708,-96.9961,8D3,ASOS,350.0,5.0,2024-01-10 12,-0.002804,0.010363,0.013167,5.4,252.8,258.1,84.7,170.1,353.0
4,42.2416,-83.6933,AASM4,COOPAB,255.0,23.0,2024-01-10 12,-0.000560,0.000004,0.000564,4.0,271.7,275.5,33.0,187.1,254.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3575,44.5272,-69.6544,WVLM1,COOPAB,21.0,49.0,2024-01-09 15,0.004361,0.017238,0.012877,7.9,273.4,282.9,65.9,453.4,17.0
3576,39.8986,-80.1656,WYNP1,COOPAB,299.0,20.0,2024-01-10 12,0.000000,0.000000,0.000000,6.8,267.6,281.5,101.6,267.6,288.0
3577,42.8202,-78.1478,WYON6,COOPAB,472.0,61.0,2024-01-10 12,0.000003,0.000003,0.000000,5.2,270.5,272.9,29.3,252.2,481.0
3578,42.7344,-71.4803,ZBWN3,COOPBC,53.0,17.0,2024-01-10 12,-0.027487,0.015237,0.042725,4.6,270.9,279.5,74.5,429.5,51.0
